# The room is the experiment
**Imaging & CNNs workshop · Laptop part 2 · in pairs · about 30 minutes**

In this part, the whole room runs one large experiment together. Every pair trains the same small CNN on the same chest X-rays. The only difference between pairs is one setting, which is written on your config card. When everyone is done, we put all the results side by side on the main screen. That way we can see what each setting really does, measured on real data instead of taken from a textbook.

You won't need to write any code. Your job is to predict what will happen, watch what actually happens, and explain the difference.

**You work in pairs, with two roles:**

- The **driver** has the keyboard and runs the cells.
- The **navigator** reads the instructions out loud, keeps an eye on the time, and makes the final call when you have to choose something.

Swap roles whenever you see 🔁.

| Step | What you do | Time |
|---|---|---|
| 0 | Start the notebook | 2 min |
| 1 | Look at the data | 5 min |
| 2 | Get your config card and make a prediction | 5 min |
| 3 | Train the model | up to 5 min |
| 4 | Read the plots and explain what happened | 10 min |
| 5 | Send your result to the room board | 1 min |
| ⭐ | Bonus: design your own experiment | if you have time left |

> ⚠️ **Before you start:** open the menu **Runtime → Change runtime type**, choose **T4 GPU** and click **Save**. A GPU is a processor that can do thousands of small calculations at the same time, which makes training much faster. Without it, training won't finish in time.

## 0 · Start the notebook
Click the ▶️ button to the left of the cell below. The cell installs one package and downloads the X-rays. This takes about a minute, so read on while it runs. When it is done, you should see a line that starts with ✅ **Ready**.

In [ ]:
#@title ▶️ Run this cell, then read on while it loads (about 1 minute)
!pip -q install medmnist
import time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from medmnist import ChestMNIST, INFO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LABELS = [v.replace('_', ' ').capitalize() for v in INFO['chestmnist']['label'].values()]
N_CLASSES = len(LABELS)
HERNIA = LABELS.index('Hernia')

def load(split):
    ds = ChestMNIST(split=split, download=True, size=28, root='.')
    return ds.imgs, torch.tensor(ds.labels, dtype=torch.float32)

raw_train, Y_train = load('train')
raw_val, Y_val = load('val')
# Rescale the pixels so they are centred around 0, which helps the network learn.
# We compute the average and the spread on the training X-rays only, because the
# validation X-rays play the role of new patients that the model has never seen.
MEAN, STD = raw_train.mean() / 255, raw_train.std() / 255
to_tensor = lambda a: ((torch.tensor(a, dtype=torch.float32) / 255 - MEAN) / STD).unsqueeze(1)
X_train, X_val = to_tensor(raw_train).to(DEVICE), to_tensor(raw_val).to(DEVICE)
Y_train, Y_val = Y_train.to(DEVICE), Y_val.to(DEVICE)
print(f'✅ Ready. Running on the {DEVICE.upper()}, with {len(X_train):,} training X-rays and {len(X_val):,} validation X-rays.')
if DEVICE == 'cpu':
    print('⚠️  There is no GPU connected. Open Runtime → Change runtime type, choose T4 GPU, and run this cell again.')

## 1 · Look at the data
The X-rays come from **ChestMNIST**. It is a shrunken version of the NIH ChestX-ray14 dataset, which is the same dataset the CheXNet paper used. The images were made very small, 28 by 28 pixels, so that a model can learn from them in a few minutes.

Radiologists' reports were used to label every X-ray with 14 possible **findings**. A finding is something a radiologist reports on an X-ray, such as an enlarged heart (*cardiomegaly*) or fluid around the lungs (*effusion*). One X-ray can show no findings, one finding, or several findings at once. A task like this, where one image can have several labels at the same time, is called a **multi-label** problem.

The first cell below shows ten random X-rays with their findings. The second cell counts how often each finding appears in the training data.

In [ ]:
#@title 👀 Show 10 random X-rays with their findings
idx = np.random.choice(len(raw_train), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, i in zip(axes.flat, idx):
    ax.imshow(raw_train[i], cmap='gray'); ax.axis('off')
    found = [LABELS[c] for c in range(N_CLASSES) if Y_train[i, c] == 1]
    ax.set_title('\n'.join(found) if found else 'No finding', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
#@title 📊 How often does each finding appear?
counts = Y_train.sum(0).cpu().numpy()
order = np.argsort(counts)
colors = ['#d62728' if i == HERNIA else '#4c72b0' for i in order]
plt.figure(figsize=(9, 5))
plt.barh([LABELS[i] for i in order], counts[order], color=colors)
for y, i in enumerate(order):
    plt.text(counts[i], y, f' {int(counts[i]):,}', va='center', fontsize=9)
plt.xlabel('number of training X-rays that show this finding'); plt.title('How many training X-rays show each finding')
plt.tight_layout(); plt.show()
no_finding = (Y_train.sum(1) == 0).float().mean().item()
print(f'{no_finding:.0%} of the training X-rays show no finding at all.')

💬 **The navigator asks the driver:** *Which finding do you think will be the hardest for the model to learn, and why?*

Remember your answer, because you will check it in step 4.

## 2 · Your config card 🔁 *swap roles*
Your instructor gives each pair a letter from A to L. Choose your letter in the dropdown menu on the right of the next cell, and run the cell. It prints a table that shows which setting is different on your card.

All pairs use the same data and the same model, and only the setting on your card changes. So if your result is different from the baseline, your setting caused the difference (apart from a little bit of randomness, which card L measures).

Your instructor has already trained config **A**, the **baseline**, on the main screen. The baseline reached an AUC of about 0.72. You will find an explanation of the AUC a little further down.

In [ ]:
#@title 🎴 Choose your config letter, then run this cell
CONFIG_ID = "A"  #@param ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L"]

BASELINE = dict(name='Baseline', lr=1e-3, batch_size=128, epochs=4, n_train=None,
                augment=False, class_weights=False, model='tiny_cnn', seed=0)
CHANGES = {
    'A': dict(),
    'B': dict(name='Huge learning rate', lr=1e-1),
    'C': dict(name='Tiny learning rate', lr=1e-5),
    'D': dict(name='Large learning rate', lr=1e-2),
    'E': dict(name='Small batches', batch_size=16),
    'F': dict(name='Huge batches', batch_size=1024),
    'G': dict(name='Little data, many epochs', n_train=2000, epochs=40),
    'H': dict(name='Class weights', class_weights=True),
    'I': dict(name='Augmentation (flip + rotate)', augment=True),
    'J': dict(name='Pretrained ResNet18 (ImageNet)', model='resnet18_pretrained'),
    'K': dict(name='ResNet18 from scratch', model='resnet18_scratch'),
    'L': dict(name='Baseline again, different seed', seed=1),
}
cfg = {**BASELINE, **CHANGES[CONFIG_ID]}

print(f'Config {CONFIG_ID}: {cfg["name"]}\n')
print(f'{"setting":<15}{"baseline":<22}{"your card":<22}')
for k in BASELINE:
    if k == 'name': continue
    mark = '  ◀ this one is different' if cfg[k] != BASELINE[k] else ''
    print(f'{k:<15}{str(BASELINE[k]):<22}{str(cfg[k]):<22}{mark}')

<details><summary><b>📖 Click here to see what each setting means.</b></summary>

| Setting | What it means | An everyday comparison |
|---|---|---|
| `lr` (learning rate) | How large a step the model's parameters take after each batch of X-rays. | Walking downhill in fog. Tiny steps get you to the bottom very slowly. Giant leaps make you jump right past the bottom of the valley. |
| `batch_size` | How many X-rays the model looks at before it takes one step. | Asking people for directions. Ask 16 people and the answer is a bit noisy, but you can ask many times. Ask 1,024 people and the answer is reliable, but it takes so long that you only get to ask a few times. |
| `epochs` | How many times the model goes through the complete training set. | Reading the same textbook again. |
| `n_train` | How many training X-rays we use. `None` means all of them, about 78,000. | The size of the textbook. |
| `augment` | During training, half of the X-rays are flipped from left to right, and every X-ray is rotated by a random angle of up to 10 degrees. The model keeps seeing slightly different versions of each image. | Looking at the same case from slightly different angles. |
| `class_weights` | Mistakes on rare findings count more in the loss than mistakes on common findings. | Giving the rare exam questions more points. |
| `model` | Either our small CNN, or ResNet18, a much larger network. ResNet18 starts either from random numbers, or from parameters that were already trained on more than a million everyday photos (the ImageNet dataset). | A first-year student, compared with an experienced photographer who has never seen an X-ray. |
| `seed` | The starting value for all random choices, such as the starting parameters and the order in which the X-rays are shown. | The same recipe, cooked on a different day. |
</details>

### What is the AUC?
We measure how good the model is with the **AUC**, which stands for *area under the ROC curve*. You don't need the formula. What matters is what it measures:

> Take one X-ray that has a certain finding and one X-ray that doesn't. The AUC is the chance that the model gives the higher score to the X-ray that has the finding.

An AUC of 0.5 means the model is guessing, like a coin flip. An AUC of 1.0 means it always puts the two X-rays in the right order. We compute the AUC for each of the 14 findings separately, and then take the average.

We use the AUC instead of accuracy because most findings are rare. A model that always says "no finding" would be right most of the time, so its accuracy would look great. Its AUC would be 0.5, which shows that it has learned nothing.

### 🔮 Predict before you train
Talk about it as a pair for two minutes, then fill in the form on the right of the next cell. The navigator makes the final call. Please don't look at the predictions of other pairs, because the experiment works best when every pair thinks for itself.

Why predict first? Once you have seen a result, it almost always looks obvious. Writing your expectation down first is the only way to find out what you really expected. The cell won't run until you have written one sentence in the `because` field.

In [ ]:
#@title 🔮 Your prediction
predicted_auc = 0.70  #@param {type:"slider", min:0.5, max:0.9, step:0.01}
predicted_vs_baseline = "About the same"  #@param ["Better than baseline", "About the same", "Worse than baseline"]
predicted_symptom = "Healthy: both losses go down"  #@param ["Healthy: both losses go down", "Loss explodes / jumps around", "Loss barely moves", "Overfitting: train loss down, val loss up", "Rare findings ignored"]
because = ""  #@param {type:"string"}
assert because.strip(), 'Please write one sentence in the "because" field first: why do you expect this result?'
print('🔒 Your prediction is saved. You can go on to step 3.')

## 3 · Train
The next cell holds the CNN you put together in part 1. There is one small difference: the `Sigmoid` at the end is missing. That is on purpose. The loss function we use, `binary_cross_entropy_with_logits`, applies the sigmoid itself before it calculates the loss. The result is the same, but the calculation is more stable.

Run the three cells below in order. The middle cell holds the training code. You don't need to read it, but you are welcome to.

The last cell trains the model. You should see one line per epoch, with the training loss, the validation loss and the validation AUC. (An **epoch** is one complete pass through all the training X-rays.) The cell stops by itself after 5 minutes at most.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # → 16 × 14 × 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # → 32 × 7 × 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 14),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
#@title ⚙️ The training code (you don't need to read this, but you are welcome to)
import torchvision

class ResNetForXrays(nn.Module):
    # ResNet18 was built for colour photos, which have 3 channels and are much larger than our X-rays.
    # So we enlarge each X-ray to 64 × 64 pixels and copy its single grayscale channel three times.
    def __init__(self, pretrained):
        super().__init__()
        self.net = torchvision.models.resnet18(weights='DEFAULT' if pretrained else None)
        self.net.fc = nn.Linear(self.net.fc.in_features, N_CLASSES)

    def forward(self, x):
        x = F.interpolate(x, size=64, mode='bilinear', align_corners=False)
        return self.net(x.repeat(1, 3, 1, 1))

def make_model(cfg):
    if cfg['model'] == 'tiny_cnn':
        return TinyCNN()
    return ResNetForXrays(pretrained=cfg['model'] == 'resnet18_pretrained')

def augment(x):
    # Flip half of the X-rays from left to right, and rotate every X-ray by a random angle
    # between -10 and +10 degrees. This runs on the GPU, so it costs almost no time.
    flip = torch.rand(len(x), device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(-1), x)
    a = (torch.rand(len(x), device=x.device) - 0.5) * 2 * np.deg2rad(10)
    zero = torch.zeros_like(a)
    theta = torch.stack([torch.stack([a.cos(), -a.sin(), zero], 1), torch.stack([a.sin(), a.cos(), zero], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

@torch.no_grad()
def evaluate(model, x, y):
    # Measure the loss (always without class weights) and the AUC of each finding.
    # This is the same yardstick for every config, so all pairs can compare their results.
    model.eval()
    logits = torch.cat([model(x[i:i + 2048]) for i in range(0, len(x), 2048)])
    loss = F.binary_cross_entropy_with_logits(logits, y).item()
    p, t = torch.sigmoid(logits).cpu().numpy(), y.cpu().numpy()
    if not np.isfinite(p).all():
        return float('nan'), float('nan'), np.full(N_CLASSES, np.nan)
    aucs = np.array([roc_auc_score(t[:, c], p[:, c]) for c in range(N_CLASSES)])
    return loss, aucs.mean(), aucs

def train(cfg, max_minutes=5):
    torch.manual_seed(cfg['seed']); np.random.seed(cfg['seed'])
    x, y = X_train, Y_train
    if cfg['n_train']:
        keep = torch.randperm(len(x), generator=torch.Generator().manual_seed(0))[:cfg['n_train']].to(DEVICE)
        x, y = x[keep], y[keep]
    # After every epoch we measure the training loss on the same 10,000 training X-rays,
    # so that the numbers from different epochs can be compared with each other.
    x_check, y_check = x[:10000], y[:10000]
    model = make_model(cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    # Class weights: when a rare finding is present, a mistake on it gets a large weight.
    # This is the same weighting as in the original CheXNet project.
    pos = y.sum(0); neg = len(y) - pos
    w_pos, w_neg = neg / (pos + neg), pos / (pos + neg)

    h = dict(cfg=cfg, step_loss=[], train_loss=[], val_loss=[], val_auc=[], class_auc=[], note='')
    t0 = time.time()
    for epoch in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(len(x), device=DEVICE)
        for i in range(0, len(x), cfg['batch_size']):
            b = perm[i:i + cfg['batch_size']]
            xb, yb = x[b], y[b]
            if cfg['augment']:
                xb = augment(xb)
            weight = yb * w_pos + (1 - yb) * w_neg if cfg['class_weights'] else None
            loss = F.binary_cross_entropy_with_logits(model(xb), yb, weight=weight)
            opt.zero_grad(); loss.backward(); opt.step()
            h['step_loss'].append(loss.item())
            if not np.isfinite(h['step_loss'][-1]):
                h['note'] = '💥 The loss became NaN ("not a number"). The numbers grew so large that training broke down.'; break
            if time.time() - t0 > max_minutes * 60:
                h['note'] = f'⏱️ Training stopped because it reached the time limit of {max_minutes} minutes.'; break
        tr_loss, _, _ = evaluate(model, x_check, y_check)
        va_loss, va_auc, cls_auc = evaluate(model, X_val, Y_val)
        for k, v in zip(['train_loss', 'val_loss', 'val_auc', 'class_auc'], [tr_loss, va_loss, va_auc, cls_auc]):
            h[k].append(v)
        print(f'epoch {epoch + 1:>2} of {cfg["epochs"]}:  training loss {tr_loss:.4f},  validation loss {va_loss:.4f},  '
              f'validation AUC {va_auc:.3f}  ({time.time() - t0:.0f} seconds so far)')
        if h['note']:
            print(h['note']); break
    return h

def diagnose(h, baseline_auc=None):
    best = int(np.nanargmax(h['val_auc'])) if not np.all(np.isnan(h['val_auc'])) else len(h['val_auc']) - 1
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    s = np.array(h['step_loss'], dtype=float)
    k = max(1, len(s) // 100)
    ax[0].plot(s, alpha=.25, color='gray'); ax[0].plot(np.convolve(s, np.ones(k) / k, mode='valid'), color='black')
    ax[0].set(title='① Training loss after every step', xlabel='step', ylabel='loss')
    if np.isfinite(s).any():
        ax[0].set_ylim(0, np.nanpercentile(s, 99) * 1.1)
    ep = np.arange(1, len(h['val_loss']) + 1)
    ax[1].plot(ep, h['train_loss'], 'o-', label='training X-rays'); ax[1].plot(ep, h['val_loss'], 's-', label='validation X-rays')
    ax[1].set(title='② Training and validation loss after every epoch', xlabel='epoch', ylabel='loss'); ax[1].legend()
    cls = h['class_auc'][best]
    colors = ['#d62728' if i == HERNIA else '#4c72b0' for i in range(N_CLASSES)]
    ax[2].barh(LABELS, cls, color=colors); ax[2].axvline(0.5, ls='--', color='gray')
    ax[2].text(0.51, N_CLASSES - 0.7, 'coin flip', color='gray', fontsize=8)
    ax[2].set(title=f'③ AUC of each finding (best epoch: {best + 1})', xlim=(0.3, 1)); ax[2].invert_yaxis()
    plt.tight_layout(); plt.show()
    h['best_auc'], h['hernia_auc'] = h['val_auc'][best], cls[HERNIA]
    h['gap'] = h['val_loss'][-1] - h['train_loss'][-1]
    print(f'Best validation AUC: {h["best_auc"]:.3f}, reached in epoch {best + 1}.')
    print(f'AUC for Hernia, the rarest finding: {h["hernia_auc"]:.3f}.')
    print(f'Validation loss minus training loss after the last epoch: {h["gap"]:+.4f}. '
          'A large positive number is a sign of overfitting.')

In [ ]:
#@title 🏃 Train your config (stops by itself after 5 minutes at most)
history = train(cfg)

## 4 · Read the plots 🔁 *swap roles*
A doctor reads a scan by looking for patterns they have seen before. You can read training plots in the same way. Run the next cell to draw three plots.

1. **Plot ① shows the training loss after every step.** The loss measures how wrong the model is on one batch of X-rays, so it should go down over time. The grey line is noisy because every batch is different. The black line is a running average, which makes the trend easier to see.
2. **Plot ② shows the training loss and the validation loss after every epoch.** The training loss is measured on X-rays the model learns from. The validation loss is measured on X-rays the model has never seen. If the two lines move apart, the model is memorising the training X-rays.
3. **Plot ③ shows the AUC of each finding.** Hernia, the rarest finding, is the red bar. The dashed line at 0.5 is the score of a coin flip.

In [ ]:
#@title 🩺 Show the plots
diagnose(history)

Now compare your plots with this table. Find the row that looks most like your plots.

| What you see | Where you see it | The most likely cause |
|---|---|---|
| The loss jumps around wildly or shoots up, and the AUC is close to 0.5. | Plot ① | The learning rate is too high. Each step is so large that the model keeps overshooting the good values and can't settle down. |
| The loss stays almost flat from the start. | Plot ① | The learning rate is too low. The steps are so small that 4 epochs are not enough to make progress. (A bug in the code can look the same.) |
| The training loss keeps going down, but the validation loss turns around and goes up. | Plot ② | Overfitting. The model has seen the same few X-rays so often that it has started to memorise them. Fewer epochs, more data or augmentation can help. |
| The red Hernia bar stays close to the 0.5 line. | Plot ③ | The finding is so rare that the model learns to ignore it. Class weights or more examples of the finding can help. |
| Both losses go down together, and most bars are clearly above 0.5. | Plots ① to ③ | Training is healthy. 🎉 Now compare your AUC with the baseline. |

**When you compare your result with other pairs, look at the AUC and not at the loss.** Here is why. Config H trains with class weights, which make mistakes on rare findings count more. That changes how its training loss is calculated, so the loss numbers of H can't be compared with yours. The AUC is calculated in the same way for every config, so it is a fair yardstick for everyone.

In [ ]:
#@title 🧾 Your diagnosis
observed_symptom = "Healthy: both losses go down"  #@param ["Healthy: both losses go down", "Loss explodes / jumps around", "Loss barely moves", "Overfitting: train loss down, val loss up", "Rare findings ignored"]
explanation = ""  #@param {type:"string"}
assert explanation.strip(), 'Please explain in one sentence what your setting did to the training.'

print(f'You predicted an AUC of {predicted_auc:.2f}. The best AUC you actually got was {history["best_auc"]:.3f}.')
print(f'You predicted: {predicted_symptom}\nYou observed:  {observed_symptom}')
print('\n🎯 Your prediction was right!' if predicted_symptom == observed_symptom
      else '\n🤔 The result surprised you. That makes it the most useful kind of result, so be ready to explain it to the room.')

💬 **Back to step 1:** was the finding you guessed really the hardest one to learn? Look at plot ③.

## 5 · Send your result to the room board
Run the next cell. It prints one line with your results, separated by commas. Copy that line, open the room board at the link below it, paste the line there and press **Add to the board**. The main screen shows the results of all pairs together, live.

In [ ]:
#@title 📮 Print your result line
ROOM_BOARD_URL = "https://claude.ai/artifact/75fDkZHAbXEjpgnMvbKKne"
line = (f'{CONFIG_ID},{cfg["name"]},{history["best_auc"]:.3f},{history["hernia_auc"]:.3f},{history["gap"]:.4f},'
        f'{predicted_auc:.2f},{predicted_symptom},{observed_symptom}')
print(line)
print(f'\nPaste the line above into the room board: {ROOM_BOARD_URL}')

## ⭐ Bonus: design experiment Z
If you finish early, you can design your own experiment. There are two rules.

1. **Change at most two settings.** If you change more, you can't tell which change caused the result.
2. **Write your hypothesis before you run anything**, for the same reason you made a prediction in step 2.

Here are some questions you could test. Does a larger learning rate rescue config F, the one with huge batches? Do class weights and a pretrained model add up? Can augmentation stop config G from overfitting?

In [ ]:
my_cfg = {**BASELINE, 'name': 'Experiment Z',
          # Change at most two settings below.
          'lr': 1e-3,
          'augment': False,
          }
hypothesis = ""   # for example: "We expect ... because ..."

assert hypothesis.strip(), 'Write your hypothesis first, then run the experiment.'
assert sum(my_cfg[k] != BASELINE[k] for k in BASELINE if k != 'name') <= 2, \
    'Change at most two settings. Otherwise you cannot tell which one caused the result.'
diagnose(train(my_cfg))

---
## Solution
Your instructor gives out the password at the end of this part. It unlocks what every card should show, with an explanation.

In [ ]:
#@title 🔑 Show the solution (the instructor gives you the password at the end)
password = ""  #@param {type:"string"}
import base64, hashlib
from IPython.display import Markdown, display
BOX = {"salt": "9296a6a2c009c558", "check": "15851c4da2fc279762213c8bacb1bc9fcba087812aceaaab428e87572f144a27", "data": "KGzAPnR9E4Gvp1fQmu3McYXwooXRHFDsz3v3pLIn/5bjqQfrKUIN0NSfUbFXOcVzj17bYfRfSJs0/b7CWL99JzcC+KrLXgysDoefBIc5/hJlfVYjhMohXsnmUVcVo7phHqzvsb8PAUZq9LGMUwn3Uo1vqj5L+sx1H2foMFhT15DSUfQv6D1rSiTevor4RKpGdCKxKyGiF8/vh5YG9qMCWR5yKHzQ1tXl1+KcUT7Pj+wuL2zXM1v3nabEeUKhfO0J3m3RlOPznZomh1WucSZTX81RhjLHb/B6VX3z4LqR2v7Of/kwP79VJb97GgAIyPlB7+fki4pBmfoCgvigH8TKSG5/2rRnkgIDRRbvkdqGgwEsIkL31TajMGwevvu7Fm+MJWWI/FV0EAXE1x4NhpirQxf8Ta1RMW265mCbqanVa2yVhTZ6JUYJ+GDXhDb/P7Mi0T8nHA661JPuWd9X4olqz4rISeAZ50P3eBUdKWgNo58CQRruqNjV476V+z0vJeIGilOa2zNm9+OeSyAvQrIenzOsxs+UaKv0OQ1gCrCjxERMTn6YC+WYnC5JFYlwL/FjFhMcGmLikhJq8oE8ZcmafGIpTixsbpqljXDC74ya0QZiLEwJt6q+FYggsvR8jnSp4KzSQh2CLN/jNVXx7uVJKpNPldW3TcnrYD8Nh11IH5mFh9Zlz0DbDZ5fgzEF2AeiutrhKvaCGbSt1Y+wME8jdZCS6w7cfCWFM0+It51vSRd7ZaqV6sEyLxFhCgN2RAkfBycrJewXYulk84XRkVO4PMHQRlHgOfFPe2sbSNsbjMYMPOayTgzjESZFwW2YKRWLvHnTfD96le1hdNEAkkoiF6CnK4qh+3V7vNjl+T7Qe+BbFqrg1xy8uq1L0x0Ww9BXnffx6pmUxwUXdRZv8JGFQkkGFR5tu8Vz6AROjCMHrpiQ1rPEpv40Z79axn4U0GZmAV8C3+EN+sc4BL9HA3eIHFlrInSHD+HhiiKjrM7Vubyxeio68xpUXq5sgWZgT7g3hxRgn5B4uaXhFFyHt5YWBjv0vpsOtFDghU3ZXrSNALaJ+hmTUg7ZHALi57j+yhB1DllJq/rGe0sSlQOUYjJOkF6Cb3pLsf8HkWhmqPsFe0CIv70dMwYTv6pK1dxwnCtb+/yo5nN75Qsqxm9O897CBksw+nMtbfrVvjOULsPiLjdQEriVz1Pk/l4cHSXFIZf8Jxwk+bBamZoDj+7h3p5zdzq0W7NKXJLKvZEILSkX0ZTnA78HiEanjhpGgn9/5B7MbxHGgx/fOzU93v/GdsfsQW5QV9L4an0EW6C0nSWqNuDre9C5d6p83gLpkxvIFqlCZSs2elj9bexw7ElAiWKsU+nH12W+ddK5hRVjxvTphZW2wKLQ1lTANg4OE88MAw7JhBjshYQg0vblMDV2jvpQ1gEHqJ71vKi325JflxrhhGkV3HXGEboWFv0Rj6/STB0R0CSyn/Ha2dd1gQ4EnDdYv2r+fh6C2++U7p5SYcghi2TYRG4Y7ipd8Lrbiyupjnqic/Y6dHgHT7sY9GpGnScX8MwU61ABWA9iRoCT1KQrOJ/KNH+iw20nO5TNT+iOvf1fuOCYqRVjDhm0zLeF2lfcgR6xa1bK20xwI+/n65BWaYtEBF0/GxkjOf4FWK3m3WjGEp/FOwtX6+f/oMtwN1bg1L5eBDyK24rEApFqRCDikSKgeh9fS902k3UFmxeuUWiRpfcIJQKMpIUMOAt2LyTxsxhzy7qtCaR5XT4zQHmvK3JoXB8GxGDt9S629ySoi2XXocv/h8Nk3GK5HV7NrBHxkQGhBOGqZP4C/gPTxp33+Vxw4u3TC+z3DTpDKAajKHBNW5xHZAubahW3kTm66BCmVyACGrk3KphTOlUhOBsiTOZ+2zoKkWSY8kQXdCwahqNL2yEkqeZDXQxnmobkO0VwBKcipIvq7X6I8sW2j2AKim3zP6/2eQe4TkAuW1DkgvSGGLSgTsq6lajgdrGS530CB5xoF0KDQvB4cqtr/pnHNQZ4syWpEsuTLcfl8oZVL6+Q7UfkBvA0wSeTvFeEVoXTyHKtKEwn3L3dJz+7usTtJRO5knumq/7L07z1OFHIF61HkCaDY+JQdCU1l9k9SRDupkmyyUiidMOtEXkyV89Q6ZWmoKGYHV1iLHuBix6KvP9EAx2yqEr1+YAPmmuwdqqwHepXZnDmSr+HqHrhC3LYrZOgmwkRC8+o88PvQVzENuVxMRBB6jYOgUe7NJ/veTDOqvkScWMQwXbcZ7l6NkfphRHDhPGzU4sxyxcOpBKX2I+o9E8h3imJim6njAUuL+/ck49ZTFoDxQL/PhasHNmaB/H5Uoax5ue9pd2v+RvkyDXsHrIThZlc6TOvN8Z4LtOJhDQ3KAboZS4NDYHQ2HMyjWQ0rrggXN2hNpVStcauX35wD5a6+64sbSFKClVWZLd9I3u9bojt3uw73VMdbXX3eTYRPmfYICR21XpFKbbu9mHIAijgXwBPlnnfHyp+0EOzew1G4lbGDHdUdz6OKL7VTIENuZZIrcARjv8ovTNp4wUVZx6sAYtPG5lOt0v6T6kYP6zt+9BrBFJbZKy+aesKT2Gegh0+O0MGiYVKrO7zQL5S/mjAMZTmnAwBJFsm633HAGeZ2V7w+eHQI1cdmlJgtkEFrIFXJNwnZ/nNeBQd7quf6hEdBBGy/dbCYH85Q1PmnNXeTiPTy7R40Pt3bw801BC29NAH0vy8FxyrfnJWtvbWijH6aq3B6MuiAwnLnE/t2lFZ0KqrjAbSRMhAOHyoj9WrIaPXwfuADKB7Hk9S7muBvWjHM44583ecagWY/qp3IOSz3XcoEky28yhx7nkY832m5NM79Hyilb6eNB/H3XNZlwbx+ZmcjA1/M0A3peYACBwfaVoz15BhWTiLZubS9S8iipOcRdQWjHHA0j7fmJOcvz5x5jbebiHcxYMpWoYu/Vy4qHvqwau6JC+N4L7SqY1aBJji1dGNUnMqiLlNYqXc4uiGLLtv4u0LrAszgdvDhqD0dN6cADAVGeK5a4+5rANhTekDYthD2qo7mSLeebTJDqj6LcM40igvg/MMMmfNlCQVw75mZ8KMKJa5bQzy504Iy6zBbMvgAkUVrRKVL6UxSRf6Dnv1kjiFh9/GCgfBpZnG2oL4QT2t8phuJvIAYB3c0U0EyBevk0w6uX0GyoT012YXbB+lQUHw4I03a7B5q/B6ddrQ8YqR+KvzHBPRDiMUZuejPO7/EKRD+LhHgzBa49lBRcVHg8gaOV2yQqOaaABxEgErR0W80GGjidETUBQzZilF2VAzmQZ9PwNRUWc73FenlpSifpdTdfqvgQG/AA4Vw8qRhoawgLk4ppz3MvtD+BSnb18S9in4/zZ8CfqSJnRsMyLf70gwFK5DsuzahuFRLFJktp36CKIeZ+ScNbVcPecM0urdwfnf24c9D8jTOBitAybNiWOcdHHh2wmExqSfNzdD+HJpIYXO74QoVjSFEjOXvLK1mw9PFIiJyvfIAZJ66MMG5DO0EoCbyYIPkyHc9llbu95TrCz3J2IkcgsSA10D8okA8Ig5cHwCpFrPWyjdVyY2m9oYvsLncp7ENEqTqrg4zgl0EFYOQIDcFmzF0ZL5+zgET8P0EiARM1LX71dccYPCYUA1fH+SexMIKAxAOXPQLlaCrj1Oib+s6LaUloEfDRcI+qCj8jtwpC4f+pZu/TO9sNlQBM/b9UCqO0lSg5Yog9Fw1zrdbOzfIft4yHoIRgeTrd6pn7aLPKjNctc+yaKWMquu/16cLcI3wuvJLDzH6OPWy1XuYunuvCipKl43lAiY8fJ76R9X08b5VKsvo8zfq8KVS6rsuerkn0ej0J4igCRiiR1efYXbXHa1LSpJAZv2vRbrqEOsJTQYZVcRVrT+djfj4W8m/ERDPofW12P0217zzCwB/53BViEKpxyaEyd4e4E2mBpvRh8k7JdcRmgzP2N8KdNBHsf6JRltVgC2izaZwYHsMwAFBY/1fkxK0ouQMn0agOHerNm0uSsc8g7DeswSsagV/o5cM7yqGr0obQ0096sFiKaqoZAPYK2ARBT7L9zQJA0h5pFs8XxFgHju8aPngPNNHZ4OwtFbtqlPhRN4AJbD/J9JGQstmZ5eScHG5i3Ef90PS3ylMlci1uSFFHDsajkK9QfgBneahUUDAAU1wqWIrVFrgBBxPJUP4DzT2pHTRc1pvJ5F+oFHzutyDSsFrpRVSkrm3TObTSPlvWsdeCsdAtoc9y4VbG1As8e1YFPUnM1pZb0jcmwSQ4ddJwe68aGW64L75a9BbtS2Axbz8OCF4HOMuPSArsEAhUEGeYRQlQ1cvQMZVfn6OaVuByZtt6099UMR6QLx6R1ISydNWm5We3JkLJofIucCMNkKcJJaRQSILvrHwLBShCFpUUSXMqSGgNUOqWIyg/gSTuEmLVRuf5sr7S+gFt7iJ5M32xsq7407QCDzMsqy0D+RjWgun5kBK21GVDO0xeJNvmKFwc9uv33+mebexy/f/eCvOLjsTDDnT5fl4KkVL9P4tLAToqIEBi5wedd5ZDpM0UUqfrdMcaAnm/rH/XEx7gwp9IFnhSuDR3T95L6hUro5R8NjiqBoP9KbaVxPlHz+TR1YZbXYsihJFM0UUIIyO2U9aYp1kibhmJ8hcu1cfGgfpfH/5bbnPOvMIAIUjkbIJu/2wR7j7B1vG1eLD2yqo8Q7iQCq2RsrwJy3LcVldjihy5Z9c9h4H+zm4Cg0qprrA+FVzjUblP/3d2F6dIktABSuFtHc4PEiQYVJCXf5BVXkTlE1/oo1kMusAuAW+OIG3bUNIzp6kaeg10Hc5/27AVSIH4nPgRJIZqvsjbha2OV0k1Yr4JjoITb/NFt3W84HOHItqTHWTCA56PQuVD6kyISrV59KIo/wGXYq3Sl7gN+fh3i1olYGsWc2ABBu68aUOrpL4btBkLCES9mC7ny0KkhWHjGDe6Ir0CTkWcFD96jra7vcVjLH17kbc9AagVjryCCmX6QTn20kpW//jHnNblbX2xQck6t1+HcANJi6qq7uGliR90Sv0ShbawK1ar8Cm0/wWez0onJ5aMisKhi6EVopSKBNI2eCiJOHsORpi0PmYwmRbDVy0GYyyhsi6YoheGfA1UL5xTh7M1uFEAI+0317zzZy9UdpJrnLHxkb1qRPU7D9dJB7OGMJ5It6L3ud4TSsDKVFnAPKJtPbiei5q5RlSRS0IX+AmmLOdjQzlcdZx9zYm32ukpHifBRAxNPc6p6YwChevJqLK+lqETKBNvpebn6Sd8WhDPVLKxSg+lu2NtBpohOVjm3Av/jCvF0YolPaGUw8fO7PtV62ktiyruwFcaAKZc47Mrt29x/MQxMKCRhqM9D+navXIv5R+I66OxKyr9wBTyDd4Y1EbojF2dzynZkIWzyi/08F1uuGU8hefpUdIpo2LmPuHddIyShfTM/hgqkq4vRmmTijStTTHeoCzLkXphRlgnuh8b1znkPDos0u4RFE9jvMptzXHlv90AXZsJs8HGBeVJP0iqYF5Z+iyVMnmWZCSfGHFHWwnxaBloflSkY7oHXO883oJCHlks9y2lXCrM0g2OW5Xso57Rgic2eAu3xnDZOFG2kvQ/FL3CanA/LbuM631iCru+AD+f+AcSCSrMiqmeBHkPXjUqxAKMPBlvvQlx4VdMHeP4ostbFiDr4lQ1K4TZAZ77VlWYa1rXKXgLDfCoYnEyteA0E2SiTUzPQw7ogX2jn42Fs/4tGqrwAQSksAPxRSTUsLVWD9iGU3KF8kLbM1fGF5WBl8lbCq3KSkx2V2grcu/D0bOWKXMs/0eKJTG7dstfQHDtKCo3IGMbApuutTLgiFy+bSGgjlTG28X3UeGx4X745+UPGcgmAzW/Y9dz5TtPtyYr1xUcgU7TTibn5uBsY36wSX6baVWCG50K+qMyRj5LYBZjs9b7mg5i7maifikLvZHM4DqktxDFsVJfMRqru596HreFKKCCAvhRRuJVGdW6PbNTVFCeizzkTXNhTsBktHrINv2mGFF8BaVwBV9lqWVSq790gl4VnQqnKYHoAQhi3TY/aTCMX7kz4DEdizLzc/kfu9FU8xaaBpsWgnrTSigzpT+aB9E6+HhqHl3xjQ0XPigZ/DEnbnwTtzNg8pBqHVpGnzs38K4Su8looFHR3dZTQR8u5E21eZJhKOg1uPa5SJgk2v9Xysru8fzPqmCnjXUM7KW+HmMUSOp2/9y1a5/7ca+2o="}

def _keystream(salt, pw, n):
    out, i = b"", 0
    while len(out) < n:
        out += hashlib.sha256(f"{salt}:{pw}:{i}".encode()).digest()
        i += 1
    return out[:n]

pw = password.strip().lower()
if not pw:
    print("Type the password in the field on the right and run this cell again.")
elif hashlib.sha256(f"{BOX['salt']}:check:{pw}".encode()).hexdigest() != BOX["check"]:
    print("That is not the right password. Check the spelling: it is all lowercase, with dashes.")
else:
    cipher = base64.b64decode(BOX["data"])
    text = bytes(a ^ b for a, b in zip(cipher, _keystream(BOX["salt"], pw, len(cipher)))).decode("utf-8")
    display(Markdown(text.replace("__CONFIG__", globals().get("CONFIG_ID", "?"))))